# AgentServe ,  GPU validation

Measures whether session-affinity routing and predictive KV-cache retention beat round-robin + LRU **on real hardware**.

Runs on free tiers. Kaggle: Accelerator → **GPU T4 x2**. Colab: Runtime → **T4 GPU** (use the `--share-gpu` cell).

Both arms use the identical gateway binary, tokenizer, and network path ,  only routing and eviction differ, so the measured delta is attributable to the scheduler.


## Setup: install vLLM and AgentServe


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install "vllm>=0.6.0" fastapi uvicorn prometheus-client httpx tiktoken redis
!git clone -q https://github.com/Parameshwaran-AA/agentserve.git
%cd agentserve
!pip -q install -e .

## Sanity check: 130 tests, no GPU needed


In [ ]:
# Prove the code is sound before spending GPU minutes on it.
!python -m pytest -q
!python scripts/gpu_validate.py --dry-run --replicas 2

## Run the A/B on 2x T4 (Kaggle)


In [ ]:
# Kaggle: 2x T4, one replica per GPU.
!python scripts/gpu_validate.py \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --replicas 2 \
    --sessions 24 --turns 10 --concurrency 8 \
    --prompt-tokens 1200 \
    --kv-budget-tokens 30000 \
    --out gpu_validation.json

## Run the A/B on 1x T4 (Colab)


In [ ]:
# Colab: 1x T4, two replicas packed onto the same card.
!python scripts/gpu_validate.py \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --replicas 2 --share-gpu \
    --max-model-len 4096 \
    --sessions 16 --turns 8 --concurrency 6 \
    --prompt-tokens 900 \
    --kv-budget-tokens 20000 \
    --out gpu_validation.json

## Read the result


In [ ]:
import json

d = json.load(open("gpu_validation.json"))
b, a = d["baseline"], d["agentserve"]
print(f"hardware : {d['hardware']}")
print(f"model    : {d['model']}  replicas={d['replicas']}  share_gpu={d['share_gpu']}")
print(f"workload : {d['sessions']} sessions, {d['calls_per_arm']} calls per arm\n")
print(f"cache hit rate   {b['cache_hit_rate']:.1%}  ->  {a['cache_hit_rate']:.1%}")
print(f"token reuse      {b['token_reuse_rate']:.1%}  ->  {a['token_reuse_rate']:.1%}")
print(f"p95 latency      {b['p95_latency_ms']:,.0f} ms  ->  {a['p95_latency_ms']:,.0f} ms")
print(f"p95 job time     {b['p95_jct_ms']:,.0f} ms  ->  {a['p95_jct_ms']:,.0f} ms")

## If both arms score the same


In [ ]:
# If both arms score the same, the cache never came under pressure -- with room
# for every session, eviction policy is irrelevant and routing is the only
# lever. Turn the screws, one at a time:
#
#   --kv-budget-tokens 12000     smaller cache, more eviction
#   --sessions 40                more sessions competing
#   --concurrency 16             more of them in flight at once
#   --prompt-tokens 2000         bigger prefixes, so each one costs more
#
# Check what fraction of capacity you are actually using first:
#   curl -s localhost:8299/debug/sessions | python -m json.tool